# BENCHMARK SO SÁNH: Vanilla LlamaGen-L (Fine-tuned 20 Epochs) vs k-Means vs MASC
**Phân bổ song song trên Kaggle 2x NVIDIA T4 (16 GB GDDR6) — Chuẩn 10-Class ImageNet-1K**

Notebook này thực hiện huấn luyện **20 Epochs** trực tiếp cho **Vanilla LlamaGen-L** (Flat vocabulary $N=16,384$, không gom cụm MASC/k-Means) để đối sánh trực tiếp với:
1. **Vanilla LlamaGen-L (Zero-shot gốc)**: FID = 45.47, IS = 8.11, Precision = 0.744
2. **k-Means Baseline (20 Epochs)**: FID = 31.42, IS = 8.85, Precision = 0.705
3. **MASC + UPipe (20 Epochs)**: FID = 33.45, IS = 8.75, Precision = 0.747
4. **VAR-d16 (20 Epochs)**: FID = 30.34, IS = 8.74, Precision = 0.684


In [ ]:
# ============================================================
# CELL 1 — Thiết lập môi trường, Clone Repositories & Kiểm tra 2x GPU T4
# ============================================================
import subprocess, os, sys, shutil, torch

print(f'Kiểm tra phần cứng: {torch.cuda.device_count()} GPU(s) khả dụng:')
for i in range(torch.cuda.device_count()):
    print(f'  [GPU {i}]: {torch.cuda.get_device_name(i)}')

WORK_DIR = '/kaggle/working'
MASC_DIR = f'{WORK_DIR}/MASC'
LLAMA_DIR = f'{WORK_DIR}/LlamaGen'
BENCHMARK_DIR = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR = f'{WORK_DIR}/weights'
CODES_DIR = f'{WORK_DIR}/codes'
CKPT_VANILLA_DIR = f'{WORK_DIR}/checkpoints_vanilla'

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(CODES_DIR, exist_ok=True)
os.makedirs(CKPT_VANILLA_DIR, exist_ok=True)

# Clone repositories nếu chưa có
def clone_repo(url, target):
    if not os.path.exists(target):
        print(f'>>> git clone {url} {target}')
        subprocess.run(f'git clone {url} {target}', shell=True, check=True)
    else:
        print(f'>>> {target} đã có sẵn, cập nhật git pull...')
        subprocess.run(f'git -C {target} pull', shell=True)

clone_repo('https://github.com/nta2112/masc-upgrade-ann', MASC_DIR)
clone_repo('https://github.com/FoundationVision/LlamaGen', LLAMA_DIR)
clone_repo('https://github.com/nta2112/Masce-baseline-benchmark', BENCHMARK_DIR)

# Cài đặt thư viện phụ trợ
subprocess.run('pip install -q pyyaml huggingface_hub webdataset accelerate scipy torch-fidelity tabulate pandas', shell=True)

for p in [MASC_DIR, LLAMA_DIR, BENCHMARK_DIR]:
    if p not in sys.path: sys.path.insert(0, p)

print('✅ Môi trường và các thư mục làm việc đã sẵn sàng!')


In [ ]:
# ============================================================
# CELL 2 — Tải Pretrained Weights LlamaGen & VQ-VAE (~1.7 GB)
# ============================================================
from huggingface_hub import hf_hub_download
import os

for fname in ['vq_ds16_c2i.pt', 'c2i_L_256.pt']:
    dest = os.path.join(WEIGHTS_DIR, fname)
    if os.path.exists(dest):
        print(f'✅ {fname} đã có sẵn ({os.path.getsize(dest)/1e6:.1f} MB), bỏ qua tải.')
        continue
    print(f'Đang tải {fname} từ FoundationVision/LlamaGen...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename=fname, local_dir=WEIGHTS_DIR)
    print(f'✅ Đã tải: {fname} ({os.path.getsize(dest)/1e6:.1f} MB)')

print('Tất cả trọng số nền tảng đã sẵn sàng trên đĩa!')


In [ ]:
# ============================================================
# CELL 3 — Chuẩn bị 10 Class Token Codes & Reference Batch (1,000 ảnh)
# ============================================================
import os, glob, io, tarfile, time, shutil, subprocess
import numpy as np, torch
from pathlib import Path
from huggingface_hub import snapshot_download

existing_npy = sorted(Path(CODES_DIR).glob('shard_*.npy'))
labels_path  = f'{CODES_DIR}/labels.npy'
REF_NPZ      = f'{WORK_DIR}/ref_10class_balanced_1000.npz'

# Xóa cache file cũ nếu có để tránh dùng lại file chỉ chứa 1 class cá
old_ref = f'{WORK_DIR}/ref_10class_1000.npz'
if os.path.exists(old_ref):
    try: os.remove(old_ref)
    except Exception: pass

# 1. Trích xuất 10,000 samples token codes
if existing_npy and os.path.exists(labels_path) and len(np.load(labels_path)) == 10000:
    print('✅ Đã có sẵn 10,000 ảnh (10 classes x 1,000 ảnh) trên đĩa!')
else:
    WDS_DIR = f'{WORK_DIR}/wds_cache'
    os.makedirs(WDS_DIR, exist_ok=True)
    print('Đang tải imagenet-llamagen-cache để lọc 10 class...')
    snapshot_download(
        repo_id='Efficient-Large-Model/imagenet-llamagen-cache',
        repo_type='dataset', local_dir=WDS_DIR,
        ignore_patterns=['*.json', '*.yaml', 'README*'],
    )
    tar_files = sorted(Path(WDS_DIR).glob('**/*.tar'))
    tar_path = tar_files[0]
    print(f'Đang trích xuất 10 class từ {tar_path.name}...')

    import urllib.request, json
    u = 'https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json'
    d = json.loads(urllib.request.urlopen(u).read().decode())
    SYNSET_TO_ID = {d[str(i)][0]: i for i in range(1000)}

    TARGET_CLASSES = list(range(10))
    MAX_PER_CLASS  = 1000
    class_counts   = {c: 0 for c in TARGET_CLASSES}
    shard_codes, all_labels = [], []
    shard_idx, total_imgs = 0, 0
    SHARD_SIZE = 512

    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf:
            if not member.name.endswith('.npz'): continue
            parts = member.name.split('/')
            if len(parts) < 2: continue
            label = SYNSET_TO_ID.get(parts[-2], -1)
            if label not in class_counts or class_counts[label] >= MAX_PER_CLASS: continue

            f = tf.extractfile(member)
            if f is None: continue
            try:
                npz = np.load(io.BytesIO(f.read()))
                codes = npz['indices'].flatten().astype(np.int32)
            except Exception: continue
            if len(codes) != 256: continue

            shard_codes.append(codes)
            all_labels.append(label)
            class_counts[label] += 1
            total_imgs += 1

            if len(shard_codes) >= SHARD_SIZE:
                np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
                shard_codes = []; shard_idx += 1
            if total_imgs >= 10000: break

    if shard_codes:
        np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
    np.save(labels_path, np.array(all_labels, dtype=np.int32))
    shutil.rmtree(WDS_DIR, ignore_errors=True)
    print(f'✅ Đã trích xuất đủ 10,000 samples token codes!')

# 2. Tạo Reference Batch (1,000 ảnh thật chuẩn cân bằng 10 class) nếu chưa có
if not os.path.exists(REF_NPZ):
    print('Đang tạo Reference Batch 10-Class CÂN BẰNG (100 ảnh/class x 10 class = 1,000 ảnh)...')
    if LLAMA_DIR not in sys.path: sys.path.insert(0, LLAMA_DIR)
    from tokenizer.tokenizer_image.vq_model import VQ_models
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    vq = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8).to(device).eval()
    st = torch.load(f'{WEIGHTS_DIR}/vq_ds16_c2i.pt', map_location='cpu')
    vq.load_state_dict(st.get('model', st))

    shards = sorted(Path(CODES_DIR).glob('shard_*.npy'))
    all_codes = np.concatenate([np.load(f) for f in shards], axis=0)

    if os.path.exists(labels_path):
        all_labels = np.load(labels_path)
        ref_indices = []
        for c in range(10):
            c_idxs = np.where(all_labels == c)[0][:100]
            ref_indices.extend(c_idxs)
        ref_indices = np.array(ref_indices)
        ref_codes = all_codes[ref_indices]
        print(f'   -> Đã trích xuất chính xác 100 ảnh/class cho 10 class (Tổng: {len(ref_codes)} ảnh)')
    else:
        rng = np.random.default_rng(42)
        ref_indices = rng.choice(len(all_codes), size=1000, replace=False)
        ref_codes = all_codes[ref_indices]
        print(f'   -> Đã lấy ngẫu nhiên 1,000 ảnh trên toàn bộ shards')

    ref_imgs = []
    bs = 20
    with torch.no_grad():
        for i in range(0, len(ref_codes), bs):
            b_c = torch.tensor(ref_codes[i:i+bs], device=device, dtype=torch.long)
            imgs = vq.decode_code(b_c, [len(b_c), 8, 16, 16])
            imgs = ((imgs.clamp(-1, 1) + 1.0) / 2.0 * 255.0).permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
            ref_imgs.append(imgs)

    ref_arr = np.concatenate(ref_imgs, axis=0)
    np.savez_compressed(REF_NPZ, arr_0=ref_arr)
    print(f'✅ Reference Batch cân bằng hoàn thành: {REF_NPZ} ({ref_arr.shape})')
    del vq; torch.cuda.empty_cache()
else:
    print(f'✅ Reference Batch cân bằng đã có sẵn: {REF_NPZ}')


In [ ]:
# ============================================================
# CELL 4 — HUẤN LUYỆN 20 EPOCHS CHO VANILLA LLAMAGEN-L (FLAT N=16,384)
# Khởi chạy song song trên GPU 0 và GPU 1 bằng torchrun DDP
# ============================================================
import subprocess, os, time, torch

# Kiểm tra nếu checkpoint đã tồn tại sẵn
VANILLA_CKPT_FINAL = f'{CKPT_VANILLA_DIR}/ckpt_final.pt'
if os.path.exists(VANILLA_CKPT_FINAL):
    print(f'✅ Đã phát hiện checkpoint Vanilla LlamaGen 20-Epochs tại:\n   {VANILLA_CKPT_FINAL}')
    print('⏩ Bỏ qua bước huấn luyện, chuyển thẳng sang Cell 5 để sinh ảnh & đánh giá metrics!')
else:
    print('Khởi động huấn luyện 20 Epochs cho Vanilla LlamaGen-L (Flat N=16,384) trên 2x GPU T4...')
    
    TRAIN_SCRIPT = f'{BENCHMARK_DIR}/baselines/01_vanilla_llamagen/train_vanilla.py'
    CONFIG_PATH  = f'{BENCHMARK_DIR}/configs/vanilla_llamagen_l.yaml'
    
    env = os.environ.copy()
    env['PYTHONPATH'] = f'{MASC_DIR}:{LLAMA_DIR}:{BENCHMARK_DIR}:' + env.get('PYTHONPATH', '')
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    env['OMP_NUM_THREADS'] = '2'

    cmd = [
        'torchrun', '--nproc_per_node=2', '--master_port=29520',
        TRAIN_SCRIPT,
        '--config',       CONFIG_PATH,
        '--codes',        CODES_DIR,
        '--out',          CKPT_VANILLA_DIR,
        '--grad-ckpt',
        '--batch-size',   '8',       # 8 per GPU x 2 accum x 2 GPUs = Global batch 32
        '--grad-accum',   '2',
        '--epochs',       '20.0',    # Đúng 20 epochs tương đương MASC, k-Means, VAR!
        '--ckpt-every',   '2000'
    ]

    print('Lệnh thực thi:')
    print(' '.join(cmd))
    print('=' * 70)

    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)

    proc.wait()
    print(f'\nExit code: {proc.returncode}')
    if proc.returncode == 0:
        print('✅ HUẤN LUYỆN 20 EPOCHS CHO VANILLA LLAMAGEN HOÀN TẤT THÀNH CÔNG!')
    else:
        print('❌ Lỗi trong quá trình huấn luyện, xem log bên trên!')


In [ ]:
# ============================================================
# CELL 5 — SINH 1,000 ẢNH & ĐÁNH GIÁ 10-CLASS FID CHO VANILLA LLAMAGEN (20 EPOCHS)
# Tự động chia việc sinh ảnh đều cho cả GPU 0 và GPU 1 (Song song 2x T4)
# ============================================================
import os, glob, sys, subprocess, torch, numpy as np
from tqdm import tqdm

if LLAMA_DIR not in sys.path: sys.path.insert(0, LLAMA_DIR)

from tokenizer.tokenizer_image.vq_model import VQ_models
from autoregressive.models.gpt import GPT_models
from autoregressive.models.generate import generate

NUM_SAMPLES = 1000
BATCH_SIZE  = 16
CFG_SCALE   = 4.5
TEMP        = 0.8
TOP_K       = 40
REF_NPZ     = f'{WORK_DIR}/ref_10class_balanced_1000.npz'
VANILLA_FT_NPZ = f'{WORK_DIR}/samples_vanilla_20ep_1000.npz'

# 1. Tìm checkpoint Vanilla vừa huấn luyện 20 epochs
ckpt_candidates = [
    f'{CKPT_VANILLA_DIR}/ckpt_final.pt',
] + sorted(glob.glob(f'{CKPT_VANILLA_DIR}/ckpt_*.pt'), reverse=True)

CKPT_PATH = next((p for p in ckpt_candidates if os.path.exists(p) and not p.endswith('.tmp')), None)
if not CKPT_PATH:
    raise FileNotFoundError('Không tìm thấy checkpoint Vanilla LlamaGen! Hãy chạy Cell 4 trước.')

print(f'Sử dụng Checkpoint: {CKPT_PATH} ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)')

# 2. Sinh 1,000 ảnh song song trên cả 2 GPU
n_gpus = torch.cuda.device_count()
print(f'Khởi tạo mô hình Vanilla LlamaGen-L (Vocab=16,384) trên {n_gpus} GPU(s)...')

models_per_gpu = []
for g_idx in range(n_gpus):
    dev = f'cuda:{g_idx}'
    m = GPT_models['GPT-L'](
        vocab_size=16384,
        block_size=256,
        num_classes=1000,
        cls_token_num=1,
        model_type='c2i',
    ).to(device=dev, dtype=torch.float16)
    
    ckpt_data = torch.load(CKPT_PATH, map_location='cpu')
    raw_st = ckpt_data.get('model', ckpt_data)
    clean_st = {k.replace('module.', '').replace('._original.', '.'): v for k, v in raw_st.items()}
    m.load_state_dict(clean_st, strict=False)
    m.eval()
    models_per_gpu.append((m, dev))
    print(f'   [GPU {g_idx}]: Model nạp thành công!')

# Cân bằng hoàn hảo 10 class: đúng 100 ảnh cho mỗi class từ 0 đến 9 (khớp 1:1 với Reference Batch)
classes = np.repeat(np.arange(10), NUM_SAMPLES // 10)
samples_per_gpu = int(np.ceil(NUM_SAMPLES / n_gpus))
all_fine_codes = []

for g_idx in range(n_gpus):
    m, dev = models_per_gpu[g_idx]
    start_idx = g_idx * samples_per_gpu
    end_idx   = min((g_idx + 1) * samples_per_gpu, NUM_SAMPLES)
    sub_classes = classes[start_idx:end_idx]
    
    print(f'   [GPU {g_idx}]: Đang sinh {len(sub_classes)} ảnh...')
    with torch.no_grad():
        for b in tqdm(range((len(sub_classes) + BATCH_SIZE - 1) // BATCH_SIZE), desc=f'GPU {g_idx} Sampling'):
            cur_c = sub_classes[b * BATCH_SIZE : min((b + 1) * BATCH_SIZE, len(sub_classes))]
            codes = generate(m, torch.tensor(cur_c, device=dev), 256, cfg_scale=CFG_SCALE, temperature=TEMP, top_k=TOP_K, sample_logits=True)
            all_fine_codes.append(codes.cpu().numpy())

all_fine = np.concatenate(all_fine_codes, axis=0)[:NUM_SAMPLES]

# 3. Decode token codes trực tiếp qua VQ-VAE (không cần mapping vì Vanilla là Fine tokens N=16,384)
print('Đang giải mã token codes thành ảnh qua VQ-VAE...')
vq_model = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8).to('cuda:0').eval()
vq_st = torch.load(f'{WEIGHTS_DIR}/vq_ds16_c2i.pt', map_location='cpu')
vq_model.load_state_dict(vq_st.get('model', vq_st))

all_images = []
with torch.no_grad():
    for b in tqdm(range((NUM_SAMPLES + BATCH_SIZE - 1) // BATCH_SIZE), desc='VQ-VAE Decoding'):
        c = torch.as_tensor(all_fine[b * BATCH_SIZE : min((b + 1) * BATCH_SIZE, NUM_SAMPLES)], device='cuda:0')
        imgs = vq_model.decode_code(c, [len(c), 8, 16, 16])
        imgs = ((imgs.clamp(-1, 1) + 1.0) / 2.0 * 255.0).permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
        all_images.append(imgs)

all_images = np.concatenate(all_images, axis=0)[:NUM_SAMPLES]
np.savez_compressed(VANILLA_FT_NPZ, arr_0=all_images)
print(f'✅ Đã lưu {NUM_SAMPLES} ảnh sinh từ Vanilla LlamaGen 20-Epoch vào: {VANILLA_FT_NPZ}')

# 4. Tính toán Metrics qua Evaluator chính thức của LlamaGen
print('\n' + '='*70)
print('ĐÁNH GIÁ 10-CLASS CONDITIONAL FID CHO VANILLA LLAMAGEN (20 EPOCHS)...')
print('='*70)

del vq_model, models_per_gpu
import gc; gc.collect(); torch.cuda.empty_cache()

eval_cmd = f'CUDA_VISIBLE_DEVICES=1 python {LLAMA_DIR}/evaluations/c2i/evaluator.py {REF_NPZ} {VANILLA_FT_NPZ}'
print(f'Lệnh Evaluator: {eval_cmd}')
proc = subprocess.Popen(eval_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n✅ HOÀN TẤT ĐÁNH GIÁ VANILLA LLAMAGEN!')


In [ ]:
# ============================================================
# CELL 6 — BẢNG ĐỐI SÁNH TOÀN DIỆN: BASELINE FLAT vs K-MEANS vs MASC vs VAR
# Dữ liệu thực tế đo đạc trực tiếp trên cùng tập 10-Class ImageNet (Cùng 20 Epochs)
# ============================================================
import pandas as pd

table = [
    {
        "Phương pháp": "Vanilla LlamaGen-L (Gốc Zero-shot)",
        "Vocabulary": "N=16,384 (Flat)",
        "Fine-tuning": "Zero-shot Pretrained",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "45.47",
        "IS ↑": "8.11",
        "Precision ↑": "0.744",
        "Recall ↑": "0.149"
    },
    {
        "Phương pháp": "Vanilla LlamaGen-L (Fine-tuned)",
        "Vocabulary": "N=16,384 (Flat)",
        "Fine-tuning": "20 Epochs (Fine-tuned)",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "[Điền từ Cell 5]",
        "IS ↑": "[Điền từ Cell 5]",
        "Precision ↑": "[Điền từ Cell 5]",
        "Recall ↑": "[Điền từ Cell 5]"
    },
    {
        "Phương pháp": "k-Means Clustering Prior",
        "Vocabulary": "k=4,096 (Euclidean)",
        "Fine-tuning": "20 Epochs (Fine-tuned)",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "31.42",
        "IS ↑": "8.85",
        "Precision ↑": "0.705",
        "Recall ↑": "0.533"
    },
    {
        "Phương pháp": "MASC + UPipe (Phương pháp đề xuất)",
        "Vocabulary": "k=4,096 (Manifold)",
        "Fine-tuning": "20 Epochs (Fine-tuned)",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "33.45",
        "IS ↑": "8.75",
        "Precision ↑": "0.747",
        "Recall ↑": "0.499"
    },
    {
        "Phương pháp": "VAR-d16 (NeurIPS 2024)",
        "Vocabulary": "V=4,096 (Next-Scale)",
        "Fine-tuning": "20 Epochs (Fine-tuned)",
        "Tokens/Ảnh": "680 (10 scales)",
        "10-Class FID ↓": "30.34",
        "IS ↑": "8.74",
        "Precision ↑": "0.684",
        "Recall ↑": "0.653"
    }
]

df = pd.DataFrame(table)
print(df.to_string(index=False))

print('\n' + '='*85)
print('BẢNG MARKDOWN DÙNG CHO BÀI BÁO (LATEX / MARKDOWN REPORT):')
print('='*85)
print(df.to_markdown(index=False))
